In [1]:
                                          ##### Setup ######
# This section initializes the SageMaker session, IAM role, bucket, prefix, and pipeline parameters.

In [2]:
# Comback imports requiered
import boto3
import sagemaker

from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.parameters import (
    ParameterInteger,
    ParameterString,
    ParameterFloat,
)
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput
from sagemaker.workflow.steps import ProcessingStep, TrainingStep, TransformStep
from sagemaker.workflow.properties import PropertyFile
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.model import Model
from sagemaker.workflow.model_step import ModelStep
from sagemaker.transformer import Transformer
from sagemaker.model_metrics import MetricsSource, ModelMetrics

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [3]:
# Import required libraries
import boto3
import sagemaker
from sagemaker.session import Session
from sagemaker.workflow.parameters import (
    ParameterInteger,
    ParameterString,
    ParameterFloat,
)

In [4]:
# Initialize SageMaker pipeline session and basic configuration
pipeline_session = PipelineSession()
region = pipeline_session.boto_region_name
role = sagemaker.get_execution_role()
bucket = pipeline_session.default_bucket()

prefix = "demand-forecasting/pipeline-byoc"

print("Region:", region)
print("Role:", role)
print("Bucket:", bucket)
print("Prefix:", prefix)

Region: us-east-1
Role: arn:aws:iam::988261566883:role/SageMakerStudioExecutionRole2026
Bucket: sagemaker-us-east-1-988261566883
Prefix: demand-forecasting/pipeline-byoc


In [5]:
# Define BYOC image URIs
preprocess_image_uri = "988261566883.dkr.ecr.us-east-1.amazonaws.com/ml-preprocessing:pipeline-v1"
train_image_uri = "988261566883.dkr.ecr.us-east-1.amazonaws.com/ml-training:pipeline-v1"
eval_image_uri = "988261566883.dkr.ecr.us-east-1.amazonaws.com/ml-preprocessing:pipeline-v2"

print(preprocess_image_uri)
print(train_image_uri)
print(eval_image_uri)

988261566883.dkr.ecr.us-east-1.amazonaws.com/ml-preprocessing:pipeline-v1
988261566883.dkr.ecr.us-east-1.amazonaws.com/ml-training:pipeline-v1
988261566883.dkr.ecr.us-east-1.amazonaws.com/ml-preprocessing:pipeline-v2


In [6]:
# Define pipeline parameters
processing_instance_count = ParameterInteger(
    name="ProcessingInstanceCount",
    default_value=1
)

training_instance_type = ParameterString(
    name="TrainingInstanceType",
    default_value="ml.m5.large"
)

model_approval_status = ParameterString(
    name="ModelApprovalStatus",
    default_value="PendingManualApproval"
)

input_data_uri = ParameterString(
    name="InputDataUri",
    default_value=f"s3://{bucket}/data/raw/"
)

rmse_threshold = ParameterFloat(
    name="RmseThreshold",
    default_value=50.0
)

In [7]:
                                ##### Preprocessing step #####
# This step runs the BYOC preprocessing container and produces train, validation, and test datasets.

In [8]:
# Import processing and pipeline step classes
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput
from sagemaker.workflow.steps import ProcessingStep
from sagemaker.workflow.pipeline_context import PipelineSession

In [9]:
# Define the preprocessing processor
preprocess_processor = ScriptProcessor(
    image_uri=preprocess_image_uri,
    command=["python3"],
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    sagemaker_session=pipeline_session,
)

In [10]:
# Define preprocessing step arguments
step_process_args = preprocess_processor.run(
    code="src/preprocessing/prep.py",
    inputs=[
        ProcessingInput(
            source=input_data_uri,
            destination="/opt/ml/processing/input",
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            source="/opt/ml/processing/output/train",
        ),
        ProcessingOutput(
            output_name="validation",
            source="/opt/ml/processing/output/validation",
        ),
        ProcessingOutput(
            output_name="test",
            source="/opt/ml/processing/output/test",
        ),
        ProcessingOutput(
            output_name="test_inference",
            source="/opt/ml/processing/output/test_inference",
        ),
    ],
)

/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


In [11]:
# Create preprocessing pipeline step
step_process = ProcessingStep(
    name="PreprocessData",
    step_args=step_process_args,
)

In [12]:
# Verificate
print(step_process.name)

PreprocessData


In [13]:
                                        ###### Training step  ######
# This step trains the model using the BYOC training container and the train/validation outputs from 
# the preprocessing step.

In [14]:
# Import training and pipeline step classes
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.workflow.steps import TrainingStep

In [15]:
# Define the training estimator
xgb_estimator = Estimator(
    image_uri=train_image_uri,
    role=role,
    instance_count=1,
    instance_type=training_instance_type,
    output_path=f"s3://{bucket}/{prefix}/training-output",
    sagemaker_session=pipeline_session,
)

In [16]:
# Set training hyperparameters
xgb_estimator.set_hyperparameters(
    train_file="train.csv",
    validation_file="validation.csv",
    target_col="item_cnt_month",
)

In [17]:
# Define training step arguments
step_train_args = xgb_estimator.fit(
    inputs={
        "train": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs[
                "train"
            ].S3Output.S3Uri,
            content_type="text/csv",
        ),
        "validation": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs[
                "validation"
            ].S3Output.S3Uri,
            content_type="text/csv",
        ),
    }
)

In [18]:
# Create training pipeline step
step_train = TrainingStep(
    name="TrainModel",
    step_args=step_train_args,
)

In [19]:
# Verification
print(step_train.name)

TrainModel


In [20]:
## Evaluation step
# This step evaluates the trained model on the test dataset using the BYOC evaluation container and 
# produces an evaluation report in JSON format.

In [21]:
# Import PropertyFile for evaluation output tracking
from sagemaker.workflow.properties import PropertyFile

In [22]:
# Define the evaluation processor
eval_processor = ScriptProcessor(
    image_uri=eval_image_uri,
    command=["python3"],
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    sagemaker_session=pipeline_session,
)

In [23]:
# Define evaluation report property file
evaluation_report = PropertyFile(
    name="EvaluationReport",
    output_name="evaluation",
    path="evaluation.json",
)

In [24]:
# Define evaluation step arguments
step_eval_args = eval_processor.run(
    code="src/preprocessing/evaluate.py",
    inputs=[
        ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/input/model",
        ),
        ProcessingInput(
            source=step_process.properties.ProcessingOutputConfig.Outputs[
                "test"
            ].S3Output.S3Uri,
            destination="/opt/ml/processing/input/test",
        ),
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation",
            source="/opt/ml/processing/output/evaluation",
        )
    ],
)

In [25]:
# Create evaluation pipeline step
step_eval = ProcessingStep(
    name="EvaluateModel",
    step_args=step_eval_args,
    property_files=[evaluation_report],
)

In [26]:
# Verification
print(step_eval.name)

EvaluateModel


In [27]:
                                        ##### Create model step ######
### This step creates a SageMaker model using the trained model artifacts and the BYOC serving image.

In [28]:
# Import model creation classes
from sagemaker.model import Model
from sagemaker.workflow.model_step import ModelStep

In [29]:
# Image define
serving_image_uri = "988261566883.dkr.ecr.us-east-1.amazonaws.com/ml-inference:pipeline-v1"

In [30]:
# Define SageMaker model for serving
model = Model(
    image_uri=serving_image_uri,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    role=role,
    sagemaker_session=pipeline_session,
)

In [31]:
# Define model creation step arguments
step_create_model_args = model.create(
    instance_type="ml.m5.large"
)

In [32]:
# Create model pipeline step
step_create_model = ModelStep(
    name="CreateModel",
    step_args=step_create_model_args,
)

In [33]:
# Verificate
print(step_create_model.name)

CreateModel


In [34]:
                            ##### Batch transform step  #######
## This step runs batch inference using the model created in the previous step.

In [35]:
# Import transformer and transform step classes
from sagemaker.transformer import Transformer
from sagemaker.workflow.steps import TransformStep

In [36]:
# Define batch transformer
transformer = Transformer(
    model_name=step_create_model.properties.ModelName,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{bucket}/{prefix}/batch-transform-output",
    sagemaker_session=pipeline_session,
)

In [37]:
# Define batch transform step arguments
step_transform_args = transformer.transform(
    data=step_process.properties.ProcessingOutputConfig.Outputs[
        "test_inference"
    ].S3Output.S3Uri,
    content_type="text/csv",
    split_type="Line",
)

In [38]:
# Create batch transform pipeline step
step_transform = TransformStep(
    name="BatchTransform",
    step_args=step_transform_args,
)

In [39]:
# Validate
print(step_transform.name)

BatchTransform


In [40]:
                                 ###### Register model step ########
## This step registers the trained model in SageMaker Model Registry using the evaluation metrics.

In [41]:
# Import model metrics classes
from sagemaker.model_metrics import MetricsSource, ModelMetrics

In [42]:
# Define model metrics from evaluation output
model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri=step_eval.arguments["ProcessingOutputConfig"]["Outputs"][0]["S3Output"]["S3Uri"] + "/evaluation.json",
        content_type="application/json",
    )
)

In [43]:
# Define model registration step arguments
step_register_model_args = model.register(
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.m5.large"],
    transform_instances=["ml.m5.large"],
    model_package_group_name="DemandForecastingModelPackageGroup",
    approval_status=model_approval_status,
    model_metrics=model_metrics,
)

In [44]:
# Create model registration pipeline step
step_register_model = ModelStep(
    name="RegisterModel",
    step_args=step_register_model_args,
)

In [45]:
# Verification
print(step_register_model.name)

RegisterModel


In [46]:
                                       ###### Fail step   ########
# This step fails the pipeline when the evaluation metric does not meet the required threshold.

In [47]:
from sagemaker.workflow.functions import JsonGet
from sagemaker.workflow.conditions import ConditionLessThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.fail_step import FailStep
from sagemaker.workflow.execution_variables import ExecutionVariables

In [48]:
# Read RMSE metric from evaluation report
rmse_condition = JsonGet(
    step_name=step_eval.name,
    property_file=evaluation_report,
    json_path="regression_metrics.rmse.value",
)

In [49]:
# Define fail step
step_fail = FailStep(
    name="FailPipeline",
    error_message="Model RMSE exceeded threshold.",
)

In [50]:
# Define condition step
step_condition = ConditionStep(
    name="CheckRMSECondition",
    conditions=[
        ConditionLessThanOrEqualTo(
            left=rmse_condition,
            right=rmse_threshold,
        )
    ],
    if_steps=[step_register_model, step_transform],
    else_steps=[step_fail],
)

In [51]:
                                       ##### Pipeline definition   #######
# This section assembles the full SageMaker pipeline.

In [52]:
# Imports
from sagemaker.workflow.pipeline import Pipeline

In [53]:
# Define pipeline
pipeline = Pipeline(
    name="DemandForecastingBYOCPipeline",
    parameters=[
        processing_instance_count,
        training_instance_type,
        model_approval_status,
        input_data_uri,
        rmse_threshold,
    ],
    steps=[
        step_process,
        step_train,
        step_eval,
        step_create_model,
        step_condition,
    ],
    sagemaker_session=pipeline_session,
)

In [54]:
# Verification
print(pipeline.name)

DemandForecastingBYOCPipeline


In [55]:
                                   #### Create or update the pipeline ####
pipeline.upsert(role_arn=role)
print("Pipeline upserted successfully.")

Pipeline upserted successfully.


In [56]:
                                      #### Start pipeline execution ####
execution = pipeline.start()
print("Execution ARN:", execution.arn)

Execution ARN: arn:aws:sagemaker:us-east-1:988261566883:pipeline/DemandForecastingBYOCPipeline/execution/dvldfz323jor


In [72]:
# Check execution status
print(execution.describe()["PipelineExecutionStatus"])

Succeeded


In [71]:
execution.list_steps()

[{'StepName': 'RegisterModel-RegisterModel',
  'StartTime': datetime.datetime(2026, 3, 31, 18, 45, 40, 44000, tzinfo=tzlocal()),
  'EndTime': datetime.datetime(2026, 3, 31, 18, 45, 41, 297000, tzinfo=tzlocal()),
  'StepStatus': 'Succeeded',
  'Metadata': {'RegisterModel': {'Arn': 'arn:aws:sagemaker:us-east-1:988261566883:model-package/DemandForecastingModelPackageGroup/2'}},
  'AttemptCount': 1},
 {'StepName': 'BatchTransform',
  'StartTime': datetime.datetime(2026, 3, 31, 18, 45, 40, 44000, tzinfo=tzlocal()),
  'EndTime': datetime.datetime(2026, 3, 31, 18, 50, 7, 156000, tzinfo=tzlocal()),
  'StepStatus': 'Succeeded',
  'Metadata': {'TransformJob': {'Arn': 'arn:aws:sagemaker:us-east-1:988261566883:transform-job/pipelines-dvldfz323jor-BatchTransform-rxee2HRm3s'}},
  'AttemptCount': 1},
 {'StepName': 'CheckRMSECondition',
  'StartTime': datetime.datetime(2026, 3, 31, 18, 45, 39, 316000, tzinfo=tzlocal()),
  'EndTime': datetime.datetime(2026, 3, 31, 18, 45, 39, 581000, tzinfo=tzlocal()),

In [59]:
print(f"s3://{bucket}/{prefix}/training-output")
print(f"s3://{bucket}/{prefix}/batch-transform-output")

s3://sagemaker-us-east-1-988261566883/demand-forecasting/pipeline-byoc/training-output
s3://sagemaker-us-east-1-988261566883/demand-forecasting/pipeline-byoc/batch-transform-output
